# Required Imports

In [1]:
import os
import re
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
from numpy import dot
from numpy.linalg import norm
from collections import defaultdict
from tqdm import tqdm

# -- NEW IMPORTS FOR GRAPHS --
from torch_geometric.data import Data, Dataset, Batch
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATConv, global_mean_pool, global_max_pool

## Parameters

In [2]:
# Configurazione PyG
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f'device being used: {device}')

# Configuration settings
n_classes = 40
n_epochs = 100
train_batch_size = 32
test_batch_size = 1
learning_rate = 0.0005
num_node_features = 100
num_nodes = 52
SANITIZE_PHASE = True
run_name = 'GNN_sanphase_200p' if SANITIZE_PHASE else 'GNN_rawphase_200p'
models_folder = os.path.join('./output/models_gnn_sanphase_200p/')

device being used: cpu


## Load Probe and Gallery sets (with On-the-Fly Phase Sanitization)

In [3]:
def extract_id_from_filename(filename):
    """ Estrae l'ID dal nome del file del grafo (es. da '00038m_2.pt' estrae '00038m') """
    match = re.search(r'(\d+[mf])_', filename)
    return match.group(1) if match else None

def sanitize_phase_matrix(phase_np):
    """
    Sanitizzazione standard CSI per la fase (shape [N_subcarriers, T_packets], es. [52, 200]):
    1. Phase unwrapping lungo l'asse delle 52 sottoportanti (axis=0)
    2. Rimozione del trend lineare (CFO/SFO) e offset per ciascun pacchetto tramite regressione lineare sulle sottoportanti
    3. Rimozione outlier impulsivi lungo il tempo (filtro Hampel / IQR per sottoportante)
    """
    n_sub, n_pkt = phase_np.shape
    # 1. Unwrap lungo le sottoportanti per ogni pacchetto
    unwrapped = np.unwrap(phase_np, axis=0)

    # 2. Rimozione trend lineare (slope + intercept) lungo l'indice di sottoportante k in [-26, ..., 26]
    k = np.linspace(-n_sub / 2.0, n_sub / 2.0, n_sub, dtype=np.float32)[:, None]
    k_mean = k.mean()
    k_var = np.sum((k - k_mean) ** 2) + 1e-8
    p_mean = unwrapped.mean(axis=0, keepdims=True)
    slope = np.sum((k - k_mean) * (unwrapped - p_mean), axis=0, keepdims=True) / k_var
    intercept = p_mean - slope * k_mean
    detrended = unwrapped - (slope * k + intercept)

    # 3. Clipping IQR per sottoportante lungo l'asse temporale per attenuare spike residui
    q25 = np.percentile(detrended, 25, axis=1, keepdims=True)
    q75 = np.percentile(detrended, 75, axis=1, keepdims=True)
    iqr = q75 - q25
    lower = q25 - 1.5 * iqr
    upper = q75 + 1.5 * iqr
    sanitized = np.clip(detrended, lower, upper).astype(np.float32)
    return sanitized

def load_graph_with_sanitized_phase(path, sanitize_phase=True):
    data = torch.load(path, weights_only=False)
    if sanitize_phase and data.x is not None and data.x.dim() == 2:
        # data.x ha forma [52, 2 * seq_len]: prima meta' = Amplitude, seconda meta' = Phase
        seq_len = data.x.shape[1] // 2
        amp_t = data.x[:, :seq_len].float()
        phase_np = data.x[:, seq_len:].detach().cpu().numpy().astype(np.float32)
        san_phase_np = sanitize_phase_matrix(phase_np)
        san_phase_t = torch.from_numpy(san_phase_np).to(dtype=amp_t.dtype, device=amp_t.device)
        data.x = torch.cat([amp_t, san_phase_t], dim=1)
    return data

class GraphReIDTrainDataset(Dataset):
    def __init__(self, data_root, sanitize_phase=True):
        super().__init__()
        self.data_root = data_root
        self.sanitize_phase = sanitize_phase
        self.samples, self.id_to_class = self._load_data()
        print(f"TRAIN: Trovati {len(self.samples)} grafi in {data_root} (sanitize_phase={self.sanitize_phase})")

    def _load_data(self):
        paths = []
        for dirpath, _, filenames in os.walk(self.data_root):
            for f in filenames:
                if f.endswith('.pt'):
                    paths.append(os.path.join(dirpath, f))

        unique_ids = sorted(list(set([extract_id_from_filename(p) for p in paths if extract_id_from_filename(p)])))
        id_to_class = {orig_id: idx for idx, orig_id in enumerate(unique_ids)}

        samples = []
        for p in paths:
            pid = extract_id_from_filename(p)
            if pid:
                samples.append((p, id_to_class[pid]))
        return samples, id_to_class

    def len(self):
        return len(self.samples)

    def get(self, idx):
        path, target = self.samples[idx]
        data = load_graph_with_sanitized_phase(path, sanitize_phase=self.sanitize_phase)
        data.y = torch.tensor([target], dtype=torch.long)
        data.path = path
        return data

class PKSampler(torch.utils.data.Sampler):
    def __init__(self, dataset, P=8, K=4):
        self.dataset = dataset
        self.P = P
        self.K = K
        self.batch_size = self.P * self.K

        self.id_to_indices = defaultdict(list)
        for idx in range(len(self.dataset)):
            target = self.dataset.samples[idx][1]
            self.id_to_indices[target].append(idx)
        self.unique_ids = list(self.id_to_indices.keys())

    def __iter__(self):
        random.shuffle(self.unique_ids)
        batches = []
        for i in range(0, len(self.unique_ids), self.P):
            p_ids = self.unique_ids[i:i+self.P]
            if len(p_ids) < self.P:
                break
            batch = []
            for pid in p_ids:
                indices = self.id_to_indices[pid]
                if len(indices) >= self.K:
                    batch.extend(random.sample(indices, self.K))
                else:
                    batch.extend(np.random.choice(indices, size=self.K, replace=True))
            batches.append(batch)

        random.shuffle(batches)
        for batch in batches:
            yield from batch

    def __len__(self):
        return (len(self.unique_ids) // self.P) * self.batch_size

class GraphProbeGalleryValTest(Dataset):
    def __init__(self, data_root, sanitize_phase=True):
        super().__init__()
        self.data_root = data_root
        self.sanitize_phase = sanitize_phase
        self.data = self.read_folder()
        self.samples = self.create_samples()
        print(f"TEST: Trovati {len(self.data)} file totali. Generati {len(self.samples)} campioni Probe-Gallery (sanitize_phase={self.sanitize_phase}).")

    def len(self):
        return len(self.samples)

    def get(self, idx):
        probe_pth, gallery_pths, is_match, probe_target, gallery_targets = self.samples[idx]
        probe_data = load_graph_with_sanitized_phase(probe_pth, sanitize_phase=self.sanitize_phase)
        gallery_data_list = [load_graph_with_sanitized_phase(item, sanitize_phase=self.sanitize_phase) for item in gallery_pths]
        return probe_data, gallery_data_list, is_match, probe_pth, gallery_pths, probe_target, gallery_targets

    def read_folder(self):
        paths = []
        for dirpath, _, filenames in os.walk(self.data_root):
            files = [f for f in filenames if f.endswith('.pt')]
            label = os.path.basename(os.path.normpath(dirpath))
            for item in files:
                paths.append([os.path.join(dirpath, item), label])
        return paths

    def create_samples(self):
        probes = [x[0] for x in self.data if x[1].lower() == 'probe']
        gallery = [x[0] for x in self.data if x[1].lower() == 'gallery']
        samples = []
        for probe in probes:
            probe_id = extract_id_from_filename(probe)
            if not probe_id:
                continue
            is_match, gallery_targets = [], []
            for item in gallery:
                item_id = extract_id_from_filename(item)
                if not item_id:
                    continue
                is_match.append(1 if probe_id == item_id else -1)
                gallery_targets.append(item_id)
            if len(gallery_targets) > 0:
                samples.append([probe, gallery, is_match, probe_id, gallery_targets])
        return samples

# Dataloaders Inizializzati
train_dataset = GraphReIDTrainDataset('./output/GraphData/D1/', sanitize_phase=SANITIZE_PHASE)
pk_sampler = PKSampler(train_dataset, P=8, K=4)
train_dataloader = DataLoader(train_dataset, batch_size=32, sampler=pk_sampler, drop_last=True)

test_dataset = GraphProbeGalleryValTest('./output/GraphData/D2t/', sanitize_phase=SANITIZE_PHASE)
test_dataloader = DataLoader(test_dataset, batch_size=1, shuffle=False)

TRAIN: Trovati 1360 grafi in ./output/GraphData/D1/ (sanitize_phase=True)
TEST: Trovati 1394 file totali. Generati 697 campioni Probe-Gallery (sanitize_phase=True).


# Graph Attention Network

In [4]:
class GraphAttentionNetwork(nn.Module):
    def __init__(self, in_channels, temporal_dim, hidden_channels, out_channels, heads=4):
        super().__init__()
        self.pos_emb = nn.Embedding(52, 16)

        self.temporal_conv = nn.Sequential(
            nn.Conv1d(in_channels=in_channels, out_channels=16, kernel_size=5, padding=2),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(in_channels=16, out_channels=32, kernel_size=3, padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)
        )

        self.feature_fusion = nn.Sequential(
            nn.Linear(32 + 16, temporal_dim),
            nn.ReLU()
        )

        self.conv1 = GATConv(temporal_dim, hidden_channels, heads=heads, dropout=0.3)
        self.conv2 = GATConv(hidden_channels * heads, out_channels, heads=1, concat=False, dropout=0.3)

    def forward(self, x, edge_index, batch):
        N = x.size(0)

        subcarrier_idx = (torch.arange(N, device=x.device) % 52)
        pos_features = self.pos_emb(subcarrier_idx)

        seq_len = x.shape[1] // 2
        x_seq = x.view(N, 2, seq_len)
        temp_features = self.temporal_conv(x_seq).squeeze(-1)

        node_features = torch.cat([temp_features, pos_features], dim=1)
        node_features = self.feature_fusion(node_features)

        x_g, alpha1 = self.conv1(node_features, edge_index, return_attention_weights=True)
        x_g = F.elu(x_g)

        x_g, alpha2 = self.conv2(x_g, edge_index, return_attention_weights=True)

        x_mean = global_mean_pool(x_g, batch)
        x_max = global_max_pool(x_g, batch)

        global_features = torch.cat([x_mean, x_max], dim=1)
        global_features = F.normalize(global_features, p=2, dim=1)

        return global_features, alpha2

# Graph Consistency Loss & Hard Triplet Loss

In [5]:
class GraphConsistencyTripletLoss(nn.Module):
    def __init__(self, margin=0.3, lambda_gc=0.005):
        super().__init__()
        self.ranking_loss = nn.MarginRankingLoss(margin=margin)
        self.lambda_gc = lambda_gc

    def forward(self, embeddings, attentions, targets, batch):
        n = embeddings.size(0)
        dist = torch.pow(embeddings, 2).sum(dim=1, keepdim=True).expand(n, n)
        dist = dist + dist.t()
        dist.addmm_(embeddings, embeddings.t(), beta=1, alpha=-2)
        dist = dist.clamp(min=1e-12).sqrt()

        mask = targets.expand(n, n).eq(targets.expand(n, n).t())
        dist_ap, dist_an = [], []

        for i in range(n):
            dist_ap.append(dist[i][mask[i]].max().unsqueeze(0))
            dist_an.append(dist[i][mask[i] == 0].min().unsqueeze(0))

        dist_ap = torch.cat(dist_ap)
        dist_an = torch.cat(dist_an)
        y = torch.ones_like(dist_an)
        triplet_loss = self.ranking_loss(dist_an, dist_ap, y)

        edge_index, alpha = attentions
        gc_loss = torch.mean(alpha ** 2)

        return triplet_loss + (self.lambda_gc * gc_loss)

# Training and Testing functions

In [6]:
def compute_ap(pos_ranks):
    """Calcolo corretto della Average Precision"""
    if len(pos_ranks) == 0:
        return 0.0
    ap = 0.0
    for i, pos in enumerate(pos_ranks):
        ap += (i + 1) / (pos + 1)
    return ap / len(pos_ranks)

def testing_step(model, test_dataloader):
    model.eval()

    test_dataset = test_dataloader.dataset if hasattr(test_dataloader, 'dataset') else test_dataloader
    sanitize_phase = getattr(test_dataset, 'sanitize_phase', SANITIZE_PHASE)

    if len(test_dataset.samples) == 0:
        print("\nWARNING: Test Dataset è vuoto!")
        return 0.0, 0.0, 0.0, 0.0

    # 1. ESTRAZIONE DELLA GALLERY (UNA VOLTA SOLA)
    gallery_pths = test_dataset.samples[0][1]

    print(f"\nEstrazione feature della Gallery ({len(gallery_pths)} grafi)...")
    gallery_features_list = []
    gallery_ids = []

    with torch.no_grad():
        batch_size = 32
        for i in range(0, len(gallery_pths), batch_size):
            batch_paths = gallery_pths[i:i+batch_size]
            batch_graphs = [load_graph_with_sanitized_phase(p, sanitize_phase=sanitize_phase) for p in batch_paths]
            g_batch = Batch.from_data_list(batch_graphs).to(device)

            features, _ = model(g_batch.x, g_batch.edge_index, g_batch.batch)
            gallery_features_list.append(features.cpu())

            batch_ids = g_batch.subject_id
            if isinstance(batch_ids, str):
                batch_ids = [batch_ids]
            gallery_ids.extend(batch_ids)

        gallery_features = torch.cat(gallery_features_list, dim=0)

    # 2. VALUTAZIONE DELLE PROBE
    print("\nValutazione Probes in corso...")
    samples = 0
    K = 10
    correct_at_k = np.zeros(K)
    AP_scores = []
    rank_matrix = []

    with torch.no_grad():
        for sample in tqdm(test_dataset.samples, leave=False):
            probe_pth = sample[0]
            probe_data = load_graph_with_sanitized_phase(probe_pth, sanitize_phase=sanitize_phase)
            probe_batch = Batch.from_data_list([probe_data]).to(device)

            probe_feat, _ = model(probe_batch.x, probe_batch.edge_index, probe_batch.batch)
            probe_feat = probe_feat.cpu()

            similarities = F.cosine_similarity(probe_feat, gallery_features)
            scores = [[sim.item(), gal_id] for sim, gal_id in zip(similarities, gallery_ids)]

            samples += 1
            topk = sorted(scores, key=lambda element: (element[0]), reverse=True)

            p_id = probe_batch.subject_id[0] if isinstance(probe_batch.subject_id, list) else probe_batch.subject_id
            rank_matrix.append([p_id, topk])

            pos_ranks = []
            for i, score in enumerate(topk):
                gal_id = score[1]
                if p_id == gal_id:
                    pos_ranks.append(i)

            if pos_ranks:
                AP = compute_ap(pos_ranks)
                AP_scores.append(AP)

        for idx, probe in enumerate(rank_matrix):
            p_id = probe[0]
            for i in range(K):
                gal_id = probe[1][i][1]
                if p_id == gal_id:
                    correct_at_k[i:] += 1
                    break

    rank_accs = (correct_at_k / samples) * 100.0
    mAP = np.mean(AP_scores) if AP_scores else 0.0

    print(f'Total samples: {samples}, Rank #1: {rank_accs[0]:.3f}%, Rank #2: {rank_accs[1]:.3f}%, Rank #3: {rank_accs[2]:.3f}%, Rank #4: {rank_accs[3]:.3f}%, Rank #5: {rank_accs[4]:.3f}%, Rank #6: {rank_accs[5]:.3f}%, Rank #7: {rank_accs[6]:.3f}%, Rank #8: {rank_accs[7]:.3f}%, Rank #9: {rank_accs[8]:.3f}%, Rank #10: {rank_accs[9]:.3f}%, mAP:{mAP*100.0:.3f}%')
    return correct_at_k[0]/samples, correct_at_k[4]/samples, correct_at_k[9]/samples, mAP

In [7]:
def training_model(model, train_dataloader, test_dataloader, optimizer, criterion, n_epochs):
    bestRank1 = 0.0
    os.makedirs(models_folder, exist_ok=True)

    for epoch in range(n_epochs):
        model.train()
        train_loss = 0.0

        for batch_data in tqdm(train_dataloader, leave=False, total=len(train_dataloader)):
            batch_data = batch_data.to(device)
            targets = batch_data.y.view(-1)

            features, attentions = model(batch_data.x, batch_data.edge_index, batch_data.batch)
            loss = criterion(features, attentions, targets, batch_data.batch)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        rank1, rank5, rank10, _ = testing_step(model, test_dataloader)

        if rank1 > bestRank1:
            torch.save(model.state_dict(), f'{models_folder}/best_{run_name}_state_r1.bin')
            bestRank1 = rank1

        print(f"Epoch: [{epoch+1}/{n_epochs}]| Loss: {train_loss/len(train_dataloader):.5f}")

## Model Parameters

In [8]:
model = GraphAttentionNetwork(in_channels=2, temporal_dim=128, hidden_channels=256, out_channels=256).to(device)
criterion = GraphConsistencyTripletLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

## Network Training

In [9]:
# Start training model
training_model(model, train_dataloader, test_dataloader, optimizer, criterion, n_epochs)


Estrazione feature della Gallery (697 grafi)...

Valutazione Probes in corso...


Total samples: 697, Rank #1: 50.933%, Rank #2: 56.528%, Rank #3: 59.254%, Rank #4: 61.406%, Rank #5: 63.271%, Rank #6: 65.854%, Rank #7: 67.719%, Rank #8: 69.154%, Rank #9: 69.584%, Rank #10: 70.301%, mAP:26.862%
Epoch: [1/100]| Loss: 0.32429



Estrazione feature della Gallery (697 grafi)...

Valutazione Probes in corso...


Total samples: 697, Rank #1: 56.815%, Rank #2: 63.415%, Rank #3: 65.710%, Rank #4: 66.858%, Rank #5: 69.154%, Rank #6: 70.588%, Rank #7: 72.023%, Rank #8: 74.032%, Rank #9: 75.323%, Rank #10: 76.471%, mAP:30.715%
Epoch: [2/100]| Loss: 0.31123



Estrazione feature della Gallery (697 grafi)...

Valutazione Probes in corso...


Total samples: 697, Rank #1: 62.267%, Rank #2: 67.432%, Rank #3: 70.588%, Rank #4: 72.740%, Rank #5: 74.892%, Rank #6: 77.475%, Rank #7: 78.766%, Rank #8: 80.057%, Rank #9: 81.062%, Rank #10: 82.066%, mAP:39.982%
Epoch: [3/100]| Loss: 0.30845


... (190 outputs left out)


Estrazione feature della Gallery (697 grafi)...

Valutazione Probes in corso...


Total samples: 697, Rank #1: 77.475%, Rank #2: 82.640%, Rank #3: 85.222%, Rank #4: 87.518%, Rank #5: 89.096%, Rank #6: 90.961%, Rank #7: 91.248%, Rank #8: 92.253%, Rank #9: 93.400%, Rank #10: 94.118%, mAP:70.553%
Epoch: [99/100]| Loss: 0.03921



Estrazione feature della Gallery (697 grafi)...

Valutazione Probes in corso...


Total samples: 697, Rank #1: 78.192%, Rank #2: 83.644%, Rank #3: 85.796%, Rank #4: 86.944%, Rank #5: 88.809%, Rank #6: 89.957%, Rank #7: 91.105%, Rank #8: 92.109%, Rank #9: 92.826%, Rank #10: 93.544%, mAP:72.411%
Epoch: [100/100]| Loss: 0.06764


## Performance Evaluation

In [10]:
m1 = GraphAttentionNetwork(in_channels=2, temporal_dim=128, hidden_channels=256, out_channels=256).to(device)
m1.load_state_dict(torch.load(f'{models_folder}/best_{run_name}_state_r1.bin', map_location=device))
print('Model w/ best R1:')
testing_step(m1, test_dataloader)

Model w/ best R1:

Estrazione feature della Gallery (697 grafi)...

Valutazione Probes in corso...


Total samples: 697, Rank #1: 83.644%, Rank #2: 87.948%, Rank #3: 89.383%, Rank #4: 89.957%, Rank #5: 91.105%, Rank #6: 91.966%, Rank #7: 93.113%, Rank #8: 94.118%, Rank #9: 94.548%, Rank #10: 94.978%, mAP:75.656%


(np.float64(0.836441893830703),
 np.float64(0.9110473457675753),
 np.float64(0.9497847919655668),
 np.float64(0.7565581036837445))